In [ ]:
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score, log_loss
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from evaluate import load
import nltk
from nltk.corpus import stopwords
import re
import os
from datetime import datetime

In [ ]:
# Model params 
max_words = 2000

In [ ]:
## Output folder
# Define model name and timestamp
unique_model_str = 'XGBoost_weighted' 
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')  

# Define output folders
output_base_path = 'results/'
unique_name_date = f"{unique_model_str}_max_n_{max_words}_{timestamp}"
output_folder = os.path.join(output_base_path, unique_name_date)

os.makedirs(output_folder, exist_ok=True) # Create directories
print(f"Model results will be saved in: {output_folder}") # Print directories for verification

In [ ]:
# Ensure stopwords are available
#nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Cleans and preprocesses text by removing stopwords, punctuation, and lowercasing."""
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    text = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', text.lower())  
    # Remove specific MAUDE patterns
    text = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', text)
    # Replace multiple spaces and strip leading/trailing whitespaces
    text = re.sub(r'\s+', ' ', text).strip()  

    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [ ]:
data_folder = 'data/'
data_file = 'cybersecurity_annotated_data.pq'
df = pd.read_parquet(os.path.join(data_folder, data_file))

# Extract relevant columns
X = df[['ID', 'text']].copy()
y = df['label']

# Preprocess text column
X['text'] = X['text'].astype(str).apply(preprocess_text)

X['text'] = X['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))

In [ ]:
# Initialize stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage lists
metrics_per_fold = []
predictions_per_fold = []

# Perform cross-validation
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"Processing Fold {fold + 1}")
    start_time = time.time()
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Process text features using TF-IDF
    vectorizer = TfidfVectorizer()
    X_train_text = vectorizer.fit_transform(X_train['text'])
    X_test_text = vectorizer.transform(X_test['text'])

    # class weights
    #    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    posweight = np.round(np.sum(y_train == 0) / np.sum(y_train == 1), 1)
    model = XGBClassifier(random_state=42, scale_pos_weight=posweight)
    
    # Train model
    model.fit(X_train_text, y_train)
    probabilities = model.predict_proba(X_test_text)
    predictions = probabilities.argmax(axis=-1)
    
    # Compute evaluation metrics
    metrics = {
        "eval_f1": f1_score(y_test, predictions),
        "eval_precision": precision_score(y_test, predictions),
        "eval_recall": recall_score(y_test, predictions),
        "eval_roc_auc": roc_auc_score(y_test, probabilities[:, 1]),
        "eval_accuracy": accuracy_score(y_test, predictions),
        "eval_loss": log_loss(y_test, probabilities)
    }
    
    # Save predictions
    test_df = pd.DataFrame({
        "ID": X_test['ID'].values,
        "text": X_test['text'].values,
        "label": y_test.values,
        "Probabilities": probabilities.tolist(),
        "Predicted_Label": predictions,
        "Fold": fold
    })
    
    # Store fold metrics
    fold_metrics = {
        "Fold": fold,
        "Time_Taken_Seconds": time.time() - start_time,
        **metrics
    }
    
    print(fold_metrics)
    
    metrics_per_fold.append(fold_metrics)
    predictions_per_fold.append(test_df)

# Convert results to DataFrame
metrics_df = pd.DataFrame(metrics_per_fold)
predictions_df = pd.concat(predictions_per_fold, ignore_index=True)

# Save results as Parquet
metrics_df.to_parquet(f"{output_folder}/cross_validation_metrics.pq", engine="pyarrow", index=False)
predictions_df.to_parquet(f"{output_folder}/cross_validation_predictions.pq", engine="pyarrow", index=False)

print("Cross-validation complete. Results saved to output folder.")

In [ ]:
# Show summary statistics
# Copied from this script: /!Comparing and tracking metrics on new dataset.ipynb
filter_cols = ["F1-Score", "Precision", "Recall", 'ROC AUC'] #'Time_Taken_Seconds'
metrics_df.rename(columns={"eval_f1":"F1-Score", 
                           "eval_precision":"Precision", 
                           "eval_recall":"Recall",
                           "eval_roc_auc":"ROC AUC"}, inplace=True)

# Non filtered
df_mean_std = metrics_df[filter_cols].agg(["mean", "std"]).reset_index()
# Filtered
#df_mean_std = df_results_filt[filter_cols].groupby("Model").agg(["mean", "std"]).reset_index()
df_mean_std = df_mean_std.round(2) #2
df_mean_std